#Pipeline Health

##Load monitoring tables

In [0]:
from pyspark.sql import functions as F

bronze = spark.table("workspace.bronze.transactions")
silver = spark.table("workspace.silver.transactions")
quarantine = spark.table("workspace.silver.transaction_quarantine")

stream_bronze = spark.table("workspace.bronze.streaming_transactions")
stream_silver = spark.table("workspace.silver.streaming_transactions")
stream_quarantine = spark.table("workspace.silver.streaming_quarantine")

fraud = spark.table("workspace.gold.streaming_fraud_transactions")
balances = spark.table("workspace.gold.account_balance_summary")

##Pipeline Health Summary

In [0]:
pipeline_health = spark.createDataFrame([(
    bronze.count(),
    silver.count(),
    quarantine.count(),
    stream_bronze.count(),
    stream_silver.count(),
    stream_quarantine.count(),
    fraud.count(),
    balances.count()
)],[
    "bronze_batch",
    "silver_batch",
    "batch_quarantine",
    "bronze_stream",
    "silver_stream",
    "stream_quarantine",
    "fraud_records",
    "account_balances"
])

display(pipeline_health)

bronze_batch,silver_batch,batch_quarantine,bronze_stream,silver_stream,stream_quarantine,fraud_records,account_balances
5010000,4681541,318459,60000,55996,3490,35740,135058


#SLA Monitoring

##Calculate latency

In [0]:
sla = (
    stream_bronze
    .agg(
        F.max("_ingest_ts").alias("last_ingestion")
    )
    .withColumn(
        "current_time",
        F.current_timestamp()
    )
    .withColumn(
        "latency_seconds",
        (
            F.col("current_time").cast("long")
            -
            F.col("last_ingestion").cast("long")
        )
    )
)

display(sla)

last_ingestion,current_time,latency_seconds
2026-08-22T07:02:19.637Z,2026-08-22T07:04:10.380Z,111


##SLA Status

In [0]:
sla_status = (
    sla
    .withColumn("environment", F.lit("Development (Serverless AvailableNow)"))
    .withColumn(
        "SLA_Status",
        F.when(F.col("latency_seconds") <= 60, "PASS")
         .otherwise("EXPECTED_STOPPED_STREAM")
    )
)

display(sla_status)

last_ingestion,current_time,latency_seconds,environment,SLA_Status
2026-08-22T07:02:19.637Z,2026-08-22T07:04:11.945Z,112,Development (Serverless AvailableNow),EXPECTED_STOPPED_STREAM


##Streaming Success Rate

In [0]:
success_rate = spark.sql("""
SELECT
    ROUND(
        100.0 *
        (SELECT COUNT(*) FROM workspace.silver.streaming_transactions)
        /
        (SELECT COUNT(*) FROM workspace.bronze.streaming_transactions),
        2
    ) AS success_percentage
""")

display(success_rate)

success_percentage
93.33


##Fraud Monitoring

In [0]:
display(
    spark.sql("""
        SELECT
            risk_level,
            COUNT(*) AS transaction_count
        FROM workspace.gold.streaming_fraud_transactions
        GROUP BY risk_level
        ORDER BY transaction_count DESC
    """)
)

risk_level,transaction_count
LOW,35735
MEDIUM,5


##Channel Monitoring

In [0]:
display(
    spark.sql("""
        SELECT
            channel,
            COUNT(*) AS transactions,
            CAST(SUM(amount) AS DECIMAL(18,2)) AS total_amount,
            CAST(AVG(amount) AS DECIMAL(18,2)) AS avg_amount
        FROM workspace.gold.streaming_fraud_transactions
        GROUP BY channel
        ORDER BY transactions DESC
    """)
)

channel,transactions,total_amount,avg_amount
UPI,29102,1453696595.38,49951.78
ATM,3600,182198722.33,50610.76
CARD,2262,114512018.95,50624.23
NETBANKING,776,39873148.53,51382.92


##Quarantine Monitoring

In [0]:
display(
    spark.sql("""
        SELECT
            quarantine_reason,
            COUNT(*) AS rejected_records
        FROM workspace.silver.streaming_quarantine
        GROUP BY quarantine_reason
        ORDER BY rejected_records DESC
    """)
)

quarantine_reason,rejected_records
UNSUCCESSFUL_TRANSACTION,1437
INVALID_AMOUNT,1031
INVALID_CURRENCY,1022


##Executive Dashboard

In [0]:
dashboard = spark.sql("""
SELECT
    (SELECT COUNT(*) FROM workspace.bronze.transactions) AS batch_bronze,
    (SELECT COUNT(*) FROM workspace.silver.transactions) AS batch_silver,
    (SELECT COUNT(*) FROM workspace.gold.account_balance_summary) AS accounts,
    (SELECT COUNT(*) FROM workspace.gold.streaming_fraud_transactions) AS fraud_records,
    (SELECT COUNT(*) FROM workspace.silver.streaming_quarantine) AS quarantined
""")

display(dashboard)

batch_bronze,batch_silver,accounts,fraud_records,quarantined
5010000,4681541,135058,35740,3490


##Persist Monitoring

In [0]:
monitoring_snapshot = (
    dashboard
    .withColumn("snapshot_time", F.current_timestamp())
)

(
    monitoring_snapshot
    .write
    .format("delta")
    .mode("append")
    .saveAsTable("workspace.gold.pipeline_monitoring")
)

print("Monitoring snapshot stored")

Monitoring snapshot stored
